# 51job 职位详情采集

In [3]:
!pip install undetected-chromedriver selenium webdriver-manager pandas requests -q
print('安装完成 ✓')

安装完成 ✓


In [4]:
import os

INPUT_CSV  = r'C:\Users\paink\Desktop\51job_列表采集结果.csv'
OUTPUT_CSV = r'C:\Users\paink\Desktop\51job_detail.csv'
DONE_FILE  = r'C:\Users\paink\Desktop\51job_detail_done.json'

PROXY_API_URL    = 'http://route.xiongmaodaili.com/xiongmao-web/apiPlus/vgl?secret=9c57729155c13c4e247b0a00dcb636d2&orderNo=VGL20241115150536kV4EjK07&count=1&isTxt=1&proxyType=1&validTime=0&removal=0&cityIds=&returnAccount=1'
PROXY_ROTATE_SEC = 60
WAIT_SEC         = 5
SLEEP_MIN        = 1.0
SLEEP_MAX        = 2.0

# True=清空重采  False=断点续采
RESET = False

if RESET:
    for f in [DONE_FILE, OUTPUT_CSV]:
        if os.path.exists(f):
            os.remove(f)
            print(f'已清除：{f}')
    print('✅ 已重置')
else:
    print('断点续采模式')
print(f'主表存在：{os.path.exists(INPUT_CSV)}')

断点续采模式
主表存在：True


In [5]:
import json, time, random, re, os, zipfile, tempfile
import pandas as pd
import requests as req_lib
from datetime import datetime
import undetected_chromedriver as uc

# ==============================
# 配置
# ==============================
INPUT_CSV  = r'C:\Users\paink\Desktop\51job_列表采集结果.csv'
OUTPUT_CSV = r'C:\Users\paink\Desktop\51job_detail.csv'
DONE_FILE  = r'C:\Users\paink\Desktop\51job_detail_done.json'

PROXY_API_URL    = 'http://route.xiongmaodaili.com/xiongmao-web/apiPlus/vgl?secret=9c57729155c13c4e247b0a00dcb636d2&orderNo=VGL20241115150536kV4EjK07&count=1&isTxt=1&proxyType=1&validTime=0&removal=0&cityIds=&returnAccount=1'
PROXY_ROTATE_SEC = 180
WAIT_SEC         = 5
SLEEP_MIN        = 1.0
SLEEP_MAX        = 2.0
RESET            = False

if RESET:
    for f in [DONE_FILE, OUTPUT_CSV]:
        if os.path.exists(f):
            os.remove(f)
            print('已清除：' + f)
    print('已重置')
else:
    print('断点续采模式')

# ==============================
# 工具函数
# ==============================

def get_proxy():
    try:
        resp = req_lib.get(PROXY_API_URL, timeout=10)
        text = resp.text.strip()
        print('  [代理原始返回] ' + text[:150])
        parts = text.split(':')
        if len(parts) >= 2 and re.match(r'\d+\.\d+\.\d+\.\d+', parts[0]):
            ip_port = parts[0] + ':' + parts[1]
            user    = parts[2] if len(parts) >= 4 else None
            pwd     = parts[3] if len(parts) >= 4 else None
            return ip_port, user, pwd
        data  = resp.json()
        inner = data.get('data') or data.get('result') or data
        if isinstance(inner, list): inner = inner[0]
        if isinstance(inner, dict):
            ip   = inner.get('ip') or inner.get('host', '')
            port = str(inner.get('port', ''))
            user = inner.get('account') or inner.get('user') or inner.get('username')
            pwd  = inner.get('password') or inner.get('passwd') or inner.get('pwd')
            if ip and port:
                return ip + ':' + port, user, pwd
    except Exception as e:
        print('  [代理获取失败] ' + str(e))
    return None, None, None


def test_proxy(ip_port, user=None, pwd=None, timeout=8):
    if not ip_port:
        return False
    if user and pwd:
        proxies = {'http':  'http://%s:%s@%s' % (user, pwd, ip_port),
                   'https': 'http://%s:%s@%s' % (user, pwd, ip_port)}
    else:
        proxies = {'http': 'http://' + ip_port, 'https': 'http://' + ip_port}
    try:
        r = req_lib.get('http://httpbin.org/ip', proxies=proxies, timeout=timeout)
        print('  [代理可用] 出口IP: ' + r.json().get('origin', ''))
        return True
    except Exception as e:
        print('  [代理不可用] ' + str(e))
        return False


def get_valid_proxy(retries=3):
    for i in range(retries):
        ip_port, user, pwd = get_proxy()
        if test_proxy(ip_port, user, pwd):
            return ip_port, user, pwd
        print('  重试代理 %d/%d...' % (i + 1, retries))
        time.sleep(2)
    print('  代理全部失败，使用本机IP直连')
    return None, None, None


def make_driver(ip_port=None, user=None, pwd=None):
    opts = uc.ChromeOptions()
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--ignore-certificate-errors')

    if ip_port:
        if user and pwd:
            host = ip_port.split(':')[0]
            port = ip_port.split(':')[1]
            manifest = (
                '{"version":"1.0.0","manifest_version":2,"name":"proxy_auth",'
                '"permissions":["proxy","tabs","unlimitedStorage","storage",'
                '"<all_urls>","webRequest","webRequestBlocking"],'
                '"background":{"scripts":["background.js"]},'
                '"minimum_chrome_version":"22.0.0"}'
            )
            background = (
                'var config={mode:"fixed_servers",rules:{singleProxy:{'
                'scheme:"http",host:"%s",port:%s},bypassList:["localhost"]}};'
                'chrome.proxy.settings.set({value:config,scope:"regular"},function(){});'
                'chrome.webRequest.onAuthRequired.addListener('
                'function(details){return {authCredentials:{username:"%s",password:"%s"}}},'
                '{urls:["<all_urls>"]},["blocking"]);'
            ) % (host, port, user, pwd)
            ext_dir = tempfile.mkdtemp()
            ext_zip = os.path.join(ext_dir, 'proxy_auth.zip')
            with zipfile.ZipFile(ext_zip, 'w') as zf:
                zf.writestr('manifest.json', manifest)
                zf.writestr('background.js', background)
            opts.add_extension(ext_zip)
            print('  代理(带认证): %s@%s' % (user, ip_port))
        else:
            opts.add_argument('--proxy-server=http://' + ip_port)
            print('  代理: ' + ip_port)
    else:
        print('  无代理，本机直连')

    drv = uc.Chrome(options=opts)
    drv.set_page_load_timeout(30)
    return drv


def get_page_text(driver):
    return driver.execute_script(
        "return document.body ? document.body.innerText : ''") or ''


def page_ok(txt):
    if any(k in txt[:400] for k in
           ['滑动验证', '访问验证', '别离开', 'TraceID',
            'ERR_CONNECTION', '无法访问', 'ERR_PROXY']):
        return False
    return len(txt) > 200


def extract_fields(txt):
    lines = [l.strip() for l in txt.split('\n') if l.strip()]
    job_title = ''
    skip_kw = ['首页', '前程无忧', '登录', '注册', '收藏', '分享', '举报', '返回', '搜索']
    for line in lines[:15]:
        if 2 < len(line) < 50 and not any(k in line for k in skip_kw):
            job_title = line
            break
    sal = ''
    sal_m = re.search(
        r'[\d.]+[千万][-~～][\d.]+[千万]|[\d.]+[千万]以上'
        r'|\d+[-~～]\d+[千万]|\d+K[-~～]\d+K|面议|薪资面议', txt)
    if sal_m:
        sal = sal_m.group(0)
    loc = exp = edu = ''
    for line in lines:
        if ('|' in line or '｜' in line) and 4 < len(line) < 60:
            parts = re.split(r'[|｜]', line)
            parts = [p.strip() for p in parts if p.strip()]
            if 2 <= len(parts) <= 4 and all(len(p) < 20 for p in parts):
                loc = parts[0]
                exp = parts[1] if len(parts) > 1 else ''
                edu = parts[2] if len(parts) > 2 else ''
                break
    jd = ''
    for kw in ['职位信息', '职位描述', '岗位职责', '工作职责', '工作内容']:
        idx = txt.find(kw)
        if idx != -1:
            jd = txt[idx:idx + 3000].strip()
            break
    if not jd and len(txt) > 100:
        jd = txt[:3000]
    return job_title, jd, sal, loc, exp, edu


def save_done(done):
    with open(DONE_FILE, 'w', encoding='utf-8') as f:
        json.dump(list(done), f, ensure_ascii=False)


def rotate_proxy(driver, cur_ip):
    ip_port, user, pwd = get_valid_proxy()
    try: driver.quit()
    except: pass
    time.sleep(5)
    for attempt in range(3):
        try:
            drv = make_driver(ip_port, user, pwd)
            print('\n[IP切换] %s → %s' % (cur_ip, ip_port))
            return drv, ip_port, user, pwd
        except Exception as e:
            print('\n[启动失败 attempt=%d] %s' % (attempt + 1, e))
            time.sleep(5)
    print('\n[换IP失败，本机直连]')
    return make_driver(), None, None, None


# ==============================
# 1. 读取链接
# ==============================
df_main   = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')
all_links = df_main['标题链接'].dropna().astype(str).str.strip().tolist()
all_links = [u for u in all_links if 'jobs.51job.com' in u and u.startswith('http')]
print('共 %d 条有效链接' % len(all_links))
print('示例: ' + (all_links[0] if all_links else '无'))

# ==============================
# 2. 断点续采
# ==============================
done = set(json.load(open(DONE_FILE, encoding='utf-8'))) if os.path.exists(DONE_FILE) else set()
todo = [u for u in all_links if u not in done]
print('已完成 %d 条，待采集 %d 条' % (len(done), len(todo)))

# ==============================
# 3. 启动 Chrome
# ==============================
cur_ip, cur_user, cur_pwd = get_valid_proxy()
driver      = make_driver(cur_ip, cur_user, cur_pwd)
last_rotate = time.time()
consec_err  = 0
print('Chrome 启动成功 ✓  代理：' + str(cur_ip))

# ==============================
# 4. 主采集循环
# ==============================
need_header = not (os.path.exists(OUTPUT_CSV) and os.path.getsize(OUTPUT_CSV) > 0)

try:
    for i, url in enumerate(todo):

        if time.time() - last_rotate >= PROXY_ROTATE_SEC:
            driver, cur_ip, cur_user, cur_pwd = rotate_proxy(driver, cur_ip)
            last_rotate = time.time()
            consec_err  = 0

        if consec_err >= 5:
            print('\n[连续失败%d次，强制换IP]' % consec_err)
            driver, cur_ip, cur_user, cur_pwd = rotate_proxy(driver, cur_ip)
            last_rotate = time.time()
            consec_err  = 0

        print('[%d/%d] %s' % (i + 1, len(todo), url[:70]), end=' ... ', flush=True)

        success = False
        job_title = jd = sal = loc = exp = edu = ''

        for attempt in range(2):
            try:
                driver.get(url)
                time.sleep(WAIT_SEC)
                txt = get_page_text(driver)
                if not page_ok(txt):
                    print('[验证码/错误 attempt=%d]' % (attempt + 1), end=' ', flush=True)
                    if attempt == 0:
                        driver, cur_ip, cur_user, cur_pwd = rotate_proxy(driver, cur_ip)
                        last_rotate = time.time()
                        time.sleep(4)
                        continue
                    break
                job_title, jd, sal, loc, exp, edu = extract_fields(txt)
                success = True
                break
            except Exception as e:
                print('[异常:%s]' % str(e), end=' ', flush=True)
                if attempt == 0:
                    try: driver.quit()
                    except: pass
                    driver = make_driver(cur_ip, cur_user, cur_pwd)
                    time.sleep(3)

        if success:
            row = {'标题链接': url, '岗位名称': job_title, '薪资范围_详情': sal,
                   '工作地点': loc, '工作年限': exp, '学历要求': edu,
                   '职位信息_全文': jd, '抓取时间': datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
            print('✓  [%s]  JD=%d字  薪=%s  地=%s' % (job_title, len(jd), sal, loc))
            consec_err = 0
        else:
            row = {'标题链接': url, '职位信息_全文': '[验证码拦截]',
                   '抓取时间': datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
            print('✗ 拦截')
            consec_err += 1

        pd.DataFrame([row]).to_csv(OUTPUT_CSV, mode='a', header=need_header,
                                   index=False, encoding='utf-8-sig')
        need_header = False
        done.add(url)
        save_done(done)
        time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

except KeyboardInterrupt:
    print('\n[中断] 进度已保存，重新运行可继续。')
finally:
    try: driver.quit()
    except: pass
    print('Chrome 已关闭。')

print('\n✅ 完成 → %s，共 %d 条' % (OUTPUT_CSV, len(done)))

断点续采模式
共 836 条有效链接
示例: https://jobs.51job.com/shanghai-ypq/171924257.html?s=sou_sou_soulb&t=0_0&req=76cfe122224ec491a03efa7b95084133
已完成 101 条，待采集 735 条
  [代理原始返回] 110.90.14.201:11276
  [代理不可用] HTTPConnectionPool(host='110.90.14.201', port=11276): Max retries exceeded with url: http://httpbin.org/ip (Caused by ProxyError('Unable to connect to proxy', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)))
  重试代理 1/3...
  [代理原始返回] 180.127.197.155:11552
  [代理不可用] HTTPConnectionPool(host='180.127.197.155', port=11552): Max retries exceeded with url: http://httpbin.org/ip (Caused by ProxyError('Unable to connect to proxy', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)))
  重试代理 2/3...
  [代理原始返回] 125.77.34.117:12569
  [代理不可用] HTTPConnectionPool(host='125.77.34.117', port=12569): Max retries exceeded with url: http://httpbin.org/ip (Caused by ProxyError('Unable to connect to proxy', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None)))
  重试代